# cvfoodid -- Train YOLOv8 ingredient detector on Colab

End-to-end walkthrough: clone repo -> download datasets -> convert to YOLO format -> train -> evaluate -> export TFLite (INT8) for mobile.

**Recommended runtime:** *Runtime -> Change runtime type -> T4 GPU* (free) or *L4* / *A100* (Pro+).

**Time:** ~2-3 hours for 100 epochs of YOLOv8n on FoodSeg103 + Padang Cuisine on a T4.

## 1. Clone the repo and install

In [ ]:
# If you forked the repo, change the URL below.
REPO_URL = 'https://github.com/OyenCar/cv-food-id2.git'
REPO_DIR = REPO_URL.split('/')[-1].replace('.git', '')

!git clone $REPO_URL
%cd $REPO_DIR
!pip install --quiet -e ".[ml,demo]"
!pip install --quiet kaggle

## 2. Verify GPU and Ultralytics import

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from ultralytics import YOLO
print('Ultralytics OK')

## 3. Authenticate Kaggle (for Padang & BEEI subsets)

Upload your `kaggle.json` from https://www.kaggle.com/settings/account:

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 4. Download datasets

**FoodSeg103** is the workhorse (~5k train + ~2k val images, 103 ingredient masks).
We pull it from the HuggingFace mirror `EduardoPacheco/FoodSeg103` (~1.25 GB total,
5--10 min on Colab) and re-emit it in the original SMU folder layout. The canonical
SMU URL is unreliable from Colab, so this path is the default.

**Padang Cuisine** and **Indonesian Traditional Foods** are pulled from Kaggle and
add Indonesian dish coverage. They require `kaggle.json` from the previous step --
skip the second cell below if you didn't upload one.


In [ ]:
# FoodSeg103 via HuggingFace mirror -- no auth, no SMU password.
# Idempotent: skips if data/raw/foodseg103/FoodSeg103/Images already exists.
import os
from pathlib import Path

marker = Path('data/raw/foodseg103/FoodSeg103/Images/category_id.txt')
if marker.is_file():
    print(f'FoodSeg103 already staged at {marker.parent}, skipping download.')
else:
    !python scripts/foodseg103_from_hf.py

!ls -lh data/raw/foodseg103/FoodSeg103/Images/img_dir/train | head -3
!ls -lh data/raw/foodseg103/FoodSeg103/Images/ann_dir/train | head -3
!wc -l data/raw/foodseg103/FoodSeg103/Images/category_id.txt


**Optional -- Kaggle add-ons.** If you uploaded `kaggle.json` above, the cell below
fetches Padang Cuisine + Indonesian Traditional Foods. Failures are non-fatal: training
still works on FoodSeg103 alone.


In [ ]:
# Optional: Kaggle datasets (only succeeds if ~/.kaggle/kaggle.json exists)
import os, subprocess
if os.path.isfile(os.path.expanduser('~/.kaggle/kaggle.json')):
    subprocess.run(['python', 'scripts/download_datasets.py',
                    'padang-cuisine', 'indonesian-traditional-foods'],
                   check=False)
else:
    print('No ~/.kaggle/kaggle.json found -- skipping Kaggle datasets.')
    print('FoodSeg103 alone is enough to train; Kaggle sets are bonus coverage.')


## 5. Convert masks to YOLO bbox labels

In [ ]:
!python scripts/prepare_yolo_dataset.py \
    --raw data/raw/foodseg103/FoodSeg103 \
    --out data/processed/yolo \
    --data-yaml configs/data.yaml
!ls data/processed/yolo/images/train | head
!ls data/processed/yolo/labels/train | head

## 6. Train YOLOv8n

Defaults: 100 epochs, 640 imgsz, batch 16. Adjust to your runtime.

In [ ]:
!python scripts/train_yolo.py \
    --data configs/data.yaml \
    --model yolov8n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --name cvfoodid-yolo

## 7. Evaluate

In [ ]:
from ultralytics import YOLO
model = YOLO('runs/detect/cvfoodid-yolo/weights/best.pt')
metrics = model.val(data='configs/data.yaml', imgsz=640)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)

## 8. Export to TFLite (INT8) for mobile

In [ ]:
!python scripts/export_tflite.py \
    --weights runs/detect/cvfoodid-yolo/weights/best.pt \
    --imgsz 640 \
    --calib data/processed/yolo/images/val \
    --int8
!ls -lh runs/detect/cvfoodid-yolo/weights/

## 9. Inference + nutrition computation

Continue in `02_inference_demo.ipynb`.